# Colab Runner: Swin SSL + Segmentation

This notebook runs your pipeline directly on Google Drive data.

It supports:
- quick smoke tests
- short sanity training runs
- full runs once configuration is verified

In [ ]:
# 1) Install dependencies
!pip -q install --upgrade transformers tqdm

In [ ]:
# 2) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2b) Pull latest code from GitHub (path must match REPO_ROOT in the next cell)
import subprocess
from pathlib import Path

_repo = Path("/content/drive/My Drive/Payne_lab_swin_transformer")
assert _repo.is_dir(), f"Clone not found: {_repo} — check path or mount Drive."

# If you track a feature branch, uncomment:
# subprocess.run(["git", "-C", str(_repo), "checkout", "feat/swin-upernet-training-pipeline"], check=True)

subprocess.run(["git", "-C", str(_repo), "pull"], check=True)
print("git pull OK:", _repo.resolve())

In [ ]:
# 3) Configure repo path and dataset paths
from pathlib import Path

# If your repo is in Drive, update this path if needed.
REPO_ROOT = Path('/content/drive/My Drive/Payne_lab_swin_transformer')

# Unlabeled data root used by SSL script (it recursively scans the 3 configured subfolders)
GDRIVE_ROOT = '/content/drive/My Drive/Petrographic images_ML work'

# Labeled image/mask dirs used by segmentation script
IMG_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/img'
MASK_DIR = '/content/drive/My Drive/Petrographic images_ML work/labelled images_PS/labelled images_PS/my_dataset/masks_machine'

# Output folder for checkpoints/visualizations
OUT_ROOT = '/content/drive/My Drive/Petrographic images_ML work/model_outputs_221'

print('Repo exists:', REPO_ROOT.exists())
print('IMG dir exists:', Path(IMG_DIR).exists())
print('MASK dir exists:', Path(MASK_DIR).exists())

In [ ]:
# 4) Smoke tests (recommended first)
import os

os.chdir(REPO_ROOT)

# SSL smoke test: 1 epoch, 2 steps
!python code/model_training_pipeline/swin_ssl_pretrain_221.py \
  --gdrive_root "$GDRIVE_ROOT" \
  --epochs 1 \
  --batch_size 2 \
  --num_workers 2 \
  --max_steps_per_epoch 2 \
  --output_dir "$OUT_ROOT/ssl_smoke" \
  --amp

# Segmentation dataloader smoke test
!python code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --no_train

In [ ]:
# 5) Full SSL pretraining run (recommended first stage)
# Adjust batch_size to your Colab GPU.
from pathlib import Path
import os

ssl_out = Path(f"{OUT_ROOT}/ssl_full")
ssl_out.mkdir(parents=True, exist_ok=True)
resume_path = ssl_out / "ssl_swinv2_last.pth"

resume_arg = f'--resume "{resume_path}"' if resume_path.exists() else ""
print("Resume mode:", "ON" if resume_arg else "OFF (starting fresh)")
print("SSL output:", ssl_out)

cmd = (
    "python code/model_training_pipeline/swin_ssl_pretrain_221.py "
    f"--gdrive_root \"{GDRIVE_ROOT}\" "
    "--epochs 50 "
    "--batch_size 4 "
    "--num_workers 2 "
    "--crop 512 "
    "--mask_patch 16 "
    "--mask_ratio 0.55 "
    "--save_recon_every 1 "
    "--num_recon_samples 2 "
    f"--output_dir \"{ssl_out}\" "
    "--amp "
    f"{resume_arg}"
)

print(cmd)
os.system(cmd)

In [ ]:
# 6) Full segmentation finetune run (stage 2)
# Reverted to simpler previous recipe.
!python code/model_training_pipeline/swin_training_pipeline_221.py \
  --img_dir "$IMG_DIR" \
  --mask_dir "$MASK_DIR" \
  --epochs 10 \
  --batch_size 2 \
  --crop 512 \
  --backbone_checkpoint "$OUT_ROOT/ssl_full/ssl_swinv2_best.pth" \
  --output_dir "$OUT_ROOT/seg_full_ssl_init"

In [ ]:
# 7) Show latest SSL reconstruction preview
# Previews are written under the same --output_dir you used, e.g. .../ssl_full/recon_previews.
from pathlib import Path
from IPython.display import Image as IPyImage, display

base = Path(OUT_ROOT)
candidates = [
    base / "ssl_full" / "recon_previews",
    base / "ssl_smoke" / "recon_previews",
] + list(base.rglob("recon_previews"))

seen = set()
preview_dir = None
for d in candidates:
    d = d.resolve() if d.exists() else None
    if d and d.is_dir() and d not in seen:
        seen.add(d)
        if any(d.glob("epoch_*.png")):
            preview_dir = d
            break

if preview_dir is None:
    for d in [base / "ssl_full" / "recon_previews", base / "ssl_smoke" / "recon_previews"]:
        if d.is_dir():
            preview_dir = d
            break

if preview_dir and preview_dir.exists():
    preview_files = sorted(preview_dir.glob("epoch_*.png"))
    if preview_files:
        print("Using:", preview_dir)
        print("Latest preview:", preview_files[-1])
        display(IPyImage(filename=str(preview_files[-1])))
    else:
        print("Folder exists but no epoch_*.png yet:", preview_dir, "\nRe-run SSL cell 5.")
else:
    print("No recon_previews folder found under", base)
    print("Run cell 5 first; SSL saves to <output_dir>/recon_previews/epoch_001.png by default.")